- ### Operations research

1. Continuous programming

    1.1. Linear and convex

    1.2. Non-linear

    1.3. Constrainted

2. Integer programming

3. Multi-objective optimisation

4. Stochastic optimisation

5. Metaherisitics

    5.1. genetic algorithms
    
    5.2. simulated annealing
    
    5.3. particle swarm optimisation

---

- ### Continuous programming

**Linear**

$$\min_x​ ​c^Tx\\ s.t.\ Ax\leq b.​$$

1. simplex method:

move along the edge from feasible vertex

2. interior-point method:

$$\min_x ​c^Tx−\mu\sum _{i=1}^m \log (b_i​−A_i​x)$$

3. the duality principl:

$$\max_x ​c^Tx\\ s.t.\ Ax\leq b.​$$

$$\Leftrightarrow$$

$$\min​ ​b^Ty\\ s.t.\ A^Ty\geq c.​$$

**Non-lienar**

$$\min_{x\in R^n} f(x)$$

1. gradient descent:

$$x_{k+1}=x_k-\alpha_k\nabla f(x_k)$$

2. Newton's method:

$$x_{k+1}=x_k-|\nabla^2 f(x_k)|^{-1} \nabla f(x_k)$$

**Constrainted**

$$\min_{x\in R^n} f(x)$$

$$s.t.\ g_i(x)\leq 0, \\ h_j(x)= 0.$$

1. Lagrange multipliers & KKT condition:

$$min_{x\in R^n} L(x,\lambda_i,\mu_j)=f(x)+\sum_i\lambda_i g_i(x)+\sum_j\mu_j h_j(x)$$

$$\Rightarrow \left\{\begin{array}{lll}\nabla_{x,\lambda_i,\mu_j} L(x^*,\lambda_i^*,\mu_j^*)=0\\
\lambda_i \geq 0 \text{\\ , for dual feasibilty}\end{array}\right.$$

---

- ### Integer programming

$$\min_x​ ​c^Tx​$$

$$s.t.\ g_i(x)\leq 0, \\ h_j(x)= 0.$$

1. enumeration:

list all possible lattices

2. branch and bound:

solve the problem without integer constraints, 

then solve two branches sub-problems if the variable $x_i$ is fractional 
$$x_i\leq\lfloor x_i \rfloor
 \text{  and  } x_i\geq\lceil x_i \rceil$$

---

- ### Multi-objective optimisation


$$\min​_x ​F(x)=[f_1(x),f_2(x),\dots,f_k(x)]^T​$$

$$s.t.\ g_i(x)\leq 0, \\ h_j(x)= 0.$$

1. weighted sum method:


$$\min_x​ \sum_{i=1}^kw_if_i(x)$$

where

$$\sum_{i=1}^kw_i=1,w_i\geq0$$

2. $\epsilon$-constraint method

$$\min​_x ​f_1(x)$$

$$s.t.\ ​f_2(x)\leq\epsilon_2,\\\dots,\\​f_k(x)\leq\epsilon_k,
\\g_i(x)\leq 0, \\ h_j(x)= 0.$$

3. lexicographic method


optimize $f_1$ first, then among all optimal $f_1$ optimize $f_2$, and then continue until $f_k$

---

- ### Stochastic optimisation

$$\min_x \mathbb{E}_{\xi}[f(x,\xi)]$$
$$s.t.\ \Pr {\{g_i(x,\xi)\leq0\}}\geq1-\alpha$$

1. sample average approximation:

$$\min_x \frac{1}{N}\sum_{i=1}^Nf(x,\xi_i)$$
$$s.t.\ \frac{1}{N}\sum_{j=1}^N1\{g_i(x,\xi_j)\}\geq1-\alpha$$


---

- ### Metaherisitics

$$x^{(t+1)}=MetaHeuristicStep(x^{(t)},f,randomness)$$

**genetic algorithm:**

In [76]:
import numpy as np
from sympy import latex, symbols
from IPython.display import display, Math

def genetic_algorithm(f,x_min,x_max,n_gens,pop_size,mutation_rate=0.01):
    # Initialize population
    pop = np.random.uniform(x_min, x_max, (pop_size//2*2, len(x_min)))
    # Evolution process
    for gen in range(n_gens):
        next_pop = np.empty((0, len(x_min)))
        for _ in range(pop_size // 2):
            # Selection (tournament selection)
            def selection(pop):
                idx1, idx2 = np.random.randint(0, len(pop), 2)
                return pop[idx1] if f(pop[idx1]) < f(pop[idx2]) else pop[idx2]
            parent1 = selection(pop)
            parent2 = selection(pop)

            # Crossover
            crossover_point = np.random.randint(1, len(x_min)-1)
            child1 = np.concatenate((parent1[:crossover_point], parent2[crossover_point:]))
            child2 = np.concatenate((parent2[:crossover_point], parent1[crossover_point:]))
            next_pop = np.vstack((next_pop, child1)) 
            next_pop = np.vstack((next_pop, child2))

        # Mutation
        for i in range(pop_size):
            for j in range(len(x_min)):
                if np.random.rand() < mutation_rate:
                    next_pop[i][j] += (x_max[j] - x_min[j]) * (n_gens - gen) / n_gens * np.random.normal(0, 1)
        pop = next_pop
    return pop[np.argmin([f(ind) for ind in pop])]

def f(t): # objective function to minimize
    global expr
    global variables
    variables = symbols('x1:' + str(len(t)+1)) # define symbolic variables x1:n
    expr = variables[0]**2 + variables[1]**2 + variables[2]**2 + 1
    return expr.subs(subs_dict(variables, t))

def subs_dict(variables, t): # create substitution dictionary
    return {variables[i]: t[i] for i in range(len(variables))}

x_min = np.array([-1,-1,-1])
x_max = np.array([1,1,1])
x_opt = genetic_algorithm(f,x_min,x_max,100,100)
display(Math(r"f(x)="+latex(expr)))
print("Minimal solution:\n x* =", x_opt, "\nf(x*) =", f(x_opt))

<IPython.core.display.Math object>

Minimal solution:
 x* = [-0.00315474  0.00138191  0.00787225] 
f(x*) = 1.00007383439643


**simulated annealing:**

In [4]:
import numpy as np
import math
from sympy import latex, symbols
from IPython.display import display, Math

def simulated_annealing(f,x,T0,kmax):
    for k in range(kmax-1):   
        T = T0*(1-(k+1)/kmax)    # cooling schedule
        x_ = x + T * np.random.randn(len(inital_value))  # generate neighbor of x 
        if f(x_) - f(x) <= 0: # accept new solution if better
            x = x_
        elif np.random.uniform(0,1) <= math.exp((f(x) - f(x_)) / T): # accept with certain probability
            x = x_
    return x

def f(t): # objective function to minimize
    global expr
    global variables
    variables = symbols('x1:' + str(len(t)+1)) # define symbolic variables x1:n
    expr = variables[0]**2 + variables[1]**2 + variables[2]**2 + 1
    return expr.subs(subs_dict(variables, t))

def subs_dict(variables, t): # create substitution dictionary
    return {variables[i]: t[i] for i in range(len(variables))}

inital_value = np.array([0.5,0.5,0.5])
x_opt = simulated_annealing(f,inital_value,1,1000)
display(Math(r"f(x)="+latex(expr)))
print("Minimal solution:\n x* =", x_opt, "\nf(x*) =", f(x_opt))

<IPython.core.display.Math object>

Minimal solution:
 x* = [-0.17549098 -0.09413973  0.0681766 ] 
f(x*) = 1.04430742251557


**particle swarm optimisation:**

In [24]:
import numpy as np
from sympy import latex, symbols
from IPython.display import display, Math

def particle_swarm(f,x_min,x_max,n_particles,kmax,w=0.5,c1=1,c2=1):
    particles = np.random.uniform(x_min, x_max, (n_particles, len(x_min)))
    velocities = np.random.uniform(-abs(x_max - x_min), abs(x_max - x_min), (n_particles, len(x_min)))
    person_best = particles.copy()
    global_best = particles[np.argmin([f(p) for p in particles])]

    # Optimization loop
    for k in range(kmax):
        for i in range(n_particles):
            r1, r2 = np.random.rand(), np.random.rand()
            velocities[i] = (w * velocities[i] +
                             c1 * r1 * (person_best[i] - particles[i]) +
                             c2 * r2 * (global_best - particles[i]))
            particles[i] += velocities[i]

            # Update personal best
            if f(particles[i]) < f(person_best[i]):
                person_best[i] = particles[i]

        # Update global best
        current_global_best = particles[np.argmin([f(p) for p in particles])]
        if f(current_global_best) < f(global_best):
            global_best = current_global_best
    return global_best
def f(t): # objective function to minimize
    global expr
    global variables
    variables = symbols('x1:' + str(len(t)+1)) # define symbolic variables x1:n
    expr = variables[0]**2 + variables[1]**2 + variables[2]**2 + 1
    return expr.subs(subs_dict(variables, t))

def subs_dict(variables, t): # create substitution dictionary
    return {variables[i]: t[i] for i in range(len(variables))}

x_min = np.array([-1,-1,-1])
x_max = np.array([1,1,1])
x_opt = particle_swarm(f,x_min,x_max,10,10)
display(Math(r"f(x)="+latex(expr)))
print("Minimal solution:\n x* =", x_opt, "\nf(x*) =", f(x_opt))

<IPython.core.display.Math object>

Minimal solution:
 x* = [-0.00283797  0.03078284  0.00498104] 
f(x*) = 1.00098044776241
